# Tanka 04: inline environments, linting and testing

`spec.json` is optional: `main.jsonnet` can return one or many `tanka.dev/v1alpha1` Environment
objects, which is how a single file describes dev and prod. Testing Jsonnet means formatting,
linting, assertions, and validating the rendered output.


In [ ]:
cd /source/work/tanka-lab
export HOME=/tmp
mkdir -p environments/inline && cat > environments/inline/main.jsonnet <<'JSONNET'
local web = import 'web.libsonnet';

local env(name, replicas) = {
  apiVersion: 'tanka.dev/v1alpha1',
  kind: 'Environment',
  metadata: { name: 'lab-' + name },
  spec: { apiServer: 'https://%s.example.com:6443' % name, namespace: 'web', injectLabels: true },
  data: { web: web.new({ name: 'web', env: name, image: 'traefik/whoami:v1.11.0', replicas: replicas }) },
};

[env('dev', 1), env('prod', 3)]
JSONNET
tk eval environments/inline | jq -r '.[] | .metadata.name + "  " + .spec.apiServer + "  " + .spec.namespace'


In [ ]:
cd /source/work/tanka-lab
tk show environments/inline --name lab-prod --dangerous-allow-redirect | yq 'select(.kind == "Deployment") | .spec.replicas, .metadata.labels'


Formatting and linting are the first tests: `jsonnetfmt --test` fails on unformatted files, `jsonnet-lint` catches unused variables and dubious constructs.


In [ ]:
cd /source/work/tanka-lab
jsonnetfmt --test lib/web.libsonnet environments/default/main.jsonnet environments/inline/main.jsonnet && echo formatted || (jsonnetfmt -i lib/web.libsonnet environments/*/main.jsonnet && echo reformatted)
jsonnet-lint -J vendor -J lib lib/web.libsonnet && echo lint ok


Assertions are unit tests without a framework: evaluate the library with a known context and check properties. Wire the same file into CI.


In [ ]:
cd /source/work/tanka-lab
cat > lib/web_test.jsonnet <<'JSONNET'
local web = import 'web.libsonnet';
local out = web.new({ name: 'web', env: 'test', image: 'traefik/whoami:v1.11.0', replicas: 2 });

assert out.deployment.spec.replicas == 2 : 'replicas must come from the context';
assert out.service.spec.ports[0].port == 80 : 'service must expose http on 80';
assert out.deployment.spec.template.metadata.labels.environment == 'test';
assert std.endsWith(out.deployment.spec.template.spec.containers[0].image, ':v1.11.0') : 'pinned tag';
'all assertions passed'
JSONNET
jsonnet -J vendor -J lib lib/web_test.jsonnet


In [ ]:
cd /source/work/tanka-lab
sed -i "s/replicas == 2/replicas == 5/" lib/web_test.jsonnet && (jsonnet -J vendor -J lib lib/web_test.jsonnet || echo "the test fails, as it should"); sed -i "s/replicas == 5/replicas == 2/" lib/web_test.jsonnet


The last test is the one a cluster would run: schema validation of the exported manifests. `tk diff` against a real cluster would follow in CI.


In [ ]:
cd /source/work/tanka-lab
rm -rf /tmp/inline && tk export /tmp/inline environments/inline --name lab-prod --format '{{.kind}}-{{.metadata.name}}' >/dev/null && kubeconform -strict -summary /tmp/inline/*.yaml


Try it: add an assertion that every container image carries a tag, then break it on purpose.
